# Instant Gratification!

Let's build a useful LLM solution - in a matter of minutes.

Our goal is to code a new kind of Web Browser. Give it a URL, and it will respond with a summary. The Reader's Digest of the internet!!

Before starting, be sure to have followed the instructions in the "README" file, including creating your API key with OpenAI and adding it to the `.env` file.

In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# Connecting to OpenAI

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI.

Troubleshooting if you have problems:

1. OpenAI takes a few minutes to register after you set up an account. If you receive an error about being over quota, try waiting a few minutes and try again.
2. As a fallback, replace the line `openai = OpenAI()` with `openai = OpenAI(api_key="your-key-here")` - while it's not recommended to hard code tokens in Jupyter lab, because then you can't share your lab with others, it's a workaround for now
3. Contact me! Message me or email ed@edwarddonner.com and we will get this to work.

Any concerns about API costs? See my notes in the README - costs should be minimal, and you can control it at every point.

In [2]:
# Load environment variables in a file called .env

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
openai = OpenAI()

In [3]:
# A class to represent a Webpage

class Website:
    url: str
    title: str
    text: str

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [4]:
# Let's try one out

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
We work with groundbreaking, proprietary LLMs verticalized for talent, we’ve
patented
our matching model, and our award-winning platform has happy customers and tons of press coverage.
Connect
with me for

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT4o have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [5]:
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [6]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]

In [7]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

## Time to bring it together - the API for OpenAI is very simple!

In [8]:
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [9]:
summarize("https://edwarddonner.com")

'# Summary of Edward Donner\'s Website\n\nEdward Donner\'s website showcases his interests in coding, experimenting with large language models (LLMs), and electronic music production. He is the co-founder and CTO of **Nebula.io**, a company focused on using AI to help individuals discover their potential and facilitate talent management. Previously, he founded **untapt**, an AI startup that was acquired in 2021. \n\n## Recent Posts\nThe website features several recent posts related to AI and LLMs:\n- **November 13, 2024:** Resources for mastering AI and LLM engineering.\n- **October 16, 2024:** Resources for transitioning from software engineer to AI data scientist.\n- **August 6, 2024:** Introduction to the "Outsmart LLM Arena," an innovative competition for LLMs.\n- **June 26, 2024:** Guidance on choosing the right LLM, including a toolkit and resources. \n\nEdward invites connections through his social media and offers opportunities to subscribe to his newsletter.'

In [10]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [11]:
display_summary("https://edwarddonner.com")

# Summary of Edward Donner's Website

The website, primarily personal, features Edward Donner's interests and expertise in coding and experimenting with Large Language Models (LLMs). Edward is also a DJ and involved in electronic music production. He is the co-founder and CTO of Nebula.io, a company focused on leveraging AI to help people discover their potential, particularly in recruitment.

## Recent Posts
- **November 13, 2024**: **Mastering AI and LLM Engineering – Resources**
- **October 16, 2024**: **From Software Engineer to AI Data Scientist – Resources**
- **August 6, 2024**: **Outsmart LLM Arena – A Battle of Diplomacy and Deviousness**
- **June 26, 2024**: **Choosing the Right LLM: Toolkit and Resources**

These posts appear to focus on resources and insights related to LLMs and transitioning to data science roles.

In [12]:
display_summary("https://cnn.com")

# CNN Website Summary

CNN is a leading news outlet that provides breaking news, in-depth analysis, and a variety of multimedia content covering current events globally. The website features sections on various topics, including U.S. and world news, politics, business, health, entertainment, sports, and science.

### Recent News Highlights:
- **South Korea's President** declared emergency martial law amid escalating tensions.
- **China** has banned the sale of materials for chip and battery production to the U.S.
- An unnamed major private company is conducting significant layoffs of thousands of employees.
- **Deadly strikes** by Israel have hit southern Lebanon, which tests a fragile ceasefire.
- **Ukrainian Forces** are reported to have only three seconds to respond to Russian attacks from drones.

### Additional Topics:
- Coverage on the ongoing **Ukraine-Russia War** and the **Israel-Hamas conflict**.
- Analysis of political figures and their influence on upcoming elections.
- Reports on various societal issues, including healthcare debates affecting transgender youth.

CNN also includes a section for subscriber-only content and a consumer guide for products and deals, such as Cyber Monday deals, alongside various podcasts and shows.

In [13]:
display_summary("https://anthropic.com")

# Anthropic Website Summary

Anthropic is an AI safety and research company based in San Francisco, focused on creating reliable and beneficial AI systems. They emphasize safety in AI development and have a diverse interdisciplinary team.

## Key Products
- **Claude 3.5 Sonnet**: Their latest and most advanced AI model, now available for interaction.
- **Claude for Enterprise**: A tailored AI solution for business needs, aimed at increasing efficiency and generating new revenue streams.

## Recent Announcements
- **New Model Updates**:
  - **Oct 22, 2024**: Introduction of Claude 3.5 Sonnet and Claude 3.5 Haiku, which enhance the capabilities of their AI offerings.
  - **Sep 4, 2024**: Launch of Claude for Enterprise.
  
## Research Focus
The company is involved in ongoing research related to AI safety, including recent topics like "Constitutional AI" and core views on AI safety that outline their principles and approaches. 

For more details, their offerings integrate directly with API usage for developers interested in leveraging AI technology.

In [14]:
display_summary("https://ohionomads.com")

# Ohio Nomads Summary

The Ohio Nomads is a chapter of the Family Motor Coach Association (FMCA), founded in 1978, with approximately 60 member RVs. This group fosters camaraderie among RV enthusiasts, focusing on fellowship and exploration of interesting locations throughout Ohio and nearby states. 

## Key Features
- **Rallies:** From May to October, the chapter organizes visits to various campgrounds, often coinciding with local festivals and events. Activities are planned, yet there is ample time for relaxation and socializing around campfires.
- **Membership:** Annual membership costs $20 after the first year and is available to all FMCA members in good standing. Members enjoy opportunities to explore and travel together.

## News and Announcements
- The website encourages visitors to check out the current rally schedule for upcoming camping events and activities. 

For more details, members are invited to visit their Facebook page and membership information is available on the site.

In [15]:
display_summary("https://fortelabs.com/blog/a-quest-for-self-knowledge-from-self-help-to-somatic-healing-part-iii-awakening-my-body")

# Summary of "Awakening My Body – A Quest for Self-Knowledge (Part III)"

The article, part of a series exploring personal growth and healing, delves into the author's journey of self-discovery through somatic experiences, prompted by chronic pain and disconnection from the body. Initially experiencing severe throat pain that impacted communication and social interactions, the author sought various medical and alternative therapies to find relief.

Key points include:

- **Connection to the Body:** The author reflects on childhood coping mechanisms that led to dissociation from bodily sensations, which resulted in painful emotional patterns carried into adulthood.
- **Ayahuasca Experience:** Participation in an ayahuasca ceremony marked a turning point, recognizing the importance of bodily awareness in trauma healing. The experience emphasized the necessity of physical expression in processing emotions.
- **Fascial Therapy:** Regular sessions with a skilled bodyworker revealed how emotions stored in the body could be released physically, leading to emotional and mental clarity.
- **Somatic Healing Benefits:** The author discusses the advantages of somatic work, such as quicker integration of healing and a profound connection to the body as a source of wisdom and stability.

Ultimately, the piece emphasizes the necessity of reconnecting with the body to achieve holistic healing and personal growth. The author expresses a commitment to continuing this exploration, seeking to bridge various aspects of life and experiences.

### Announcements
- **Annual Review Program:** Join their program in December and January to start 2025 with clarity and focus.

In [16]:
def count_occurrences(sentence, letter):
  """Counts the number of occurrences of a letter in a sentence.

  Args:
    sentence: The sentence to search.
    letter: The letter to count.

  Returns:
    The number of occurrences of the letter in the sentence.
  """

  count = 0
  for char in sentence:
    if char.lower() == letter.lower():
      count += 1
  return count

# Example usage:
sentence = "How many times does the letter 'a' appear in this sentence?"
letter = "a"
result = count_occurrences(sentence, letter)
print("The letter 'a' appears", result, "times in the sentence.")


The letter 'a' appears 4 times in the sentence.


In [17]:
def count_letter(sentence, letter):
    return sentence.lower().count(letter)

sentence = "How many times does the letter 'a' appear in this sentence."
letter = 'a'

count = count_letter(sentence, letter)
print(f"The letter '{letter}' appears {count} times in the sentence.")

The letter 'a' appears 4 times in the sentence.
